# Script to Evaluate DELPHI Parameter Tuning

We will tune DELPHI parameters on first month data per region and evaluate the prediction power on the next two months.

In [1]:
import numpy as np
import pandas as pd
import os
from datetime import datetime, timedelta

os.chdir("..")

In [2]:
from pandemic_functions.delphi_functions.DELPHI_model_fitting import solve_and_predict_area
from pandemic_functions.pandemic_params import *
from pandemic_functions.delphi_functions.DELPHI_utils import compute_mae, compute_mape, compute_mse

In [16]:
start_date_default = '2020-03-01'
end_date_default ='2020-06-01' 
split_default = '2020-04-01'


### Function to Perform Evaluation

In [ ]:

def evaluate_delphi_fitting(region: str, start_date: str, end_date: str, split: str, results_df: pd.DataFrame):
    country, province = region_symbol_country_dict[region]
    continent = region_symbol_continent_dict[region]
    country_sub = country.replace(" ", "_")
    province_sub = province.replace(" ", "_")

    totalcases = pd.read_csv(
                f"pandemic_functions/pandemic_data/Cases_{country_sub}_{province_sub}.csv"
            )
    totalcases.date = pd.to_datetime(totalcases.date).dt.strftime('%Y-%m-%d')
    
    day_100 = pd.to_datetime(totalcases[totalcases.day_since100 == 0].date.iloc[0])

    if day_100 > pd.to_datetime(start_date):
        shift_days = (day_100 - pd.to_datetime(start_date)).days
        start_date = str(day_100.date())
        end_date = str((pd.to_datetime(end_date) + timedelta(days=shift_days)).date())
        split = str((pd.to_datetime(split) + timedelta(days=shift_days)).date())

    yesterday = str((pd.to_datetime(split) - timedelta(days=1)).date())

    print(f"Evaluating Delphi for {country}, {province}")
    
    try:
        df_parameters, df_predictions_since_today, df_predictions_since_100, output = solve_and_predict_area(
            region, yesterday, None, None, start_date, end_date
        )
    except ValueError as err:
        print(f"Delphi parameters solve failed with error {err}")
        raise err 

    test_set = totalcases[
            (totalcases.day_since100 >= 0) &
            (totalcases.date >= str((pd.to_datetime(split)).date())) &
            (totalcases.date <= str((pd.to_datetime(end_date)).date()))
            ][["date", "day_since100", "case_cnt", "death_cnt"]].reset_index(drop=True)

    results_df = pd.concat([results_df, pd.Series(
                {'region': region, 'start_date': start_date, 'split_date': split, 'end_date': end_date, 
                   'target': 'Total Detected Cases', 
                   'MSE': compute_mse(test_set['case_cnt'], df_predictions_since_today['Total Detected']),
                   'MAE': compute_mae(test_set['case_cnt'], df_predictions_since_today['Total Detected']),
                   'MAPE': compute_mape(test_set['case_cnt'], df_predictions_since_today['Total Detected'])
                }).to_frame().T
                ], ignore_index=True, axis=0
    )

    results_df = pd.concat([results_df, pd.Series(
                {
                   'region': region, 'start_date': start_date, 'split_date': split, 'end_date': end_date, 
                   'target': 'Total Detected Deaths', 
                   'MSE': compute_mse(test_set['death_cnt'], df_predictions_since_today['Total Detected Deaths']),
                   'MAE': compute_mae(test_set['death_cnt'], df_predictions_since_today['Total Detected Deaths']),
                   'MAPE': compute_mape(test_set['death_cnt'], df_predictions_since_today['Total Detected Deaths'])
                }).to_frame().T
                ], ignore_index=True, axis=0
    )

    return results_df

### Run evaluation for all regions

In [18]:
results_df = pd.DataFrame(columns=['region', 'start_date', 'split_date', 'end_date', 'target', 'MSE', 'MAE', 'MAPE'])

for region in ['US-NY', 'US-FL', 'ES', 'BR', 'SG', 'DE']:
    try:
        results_df = evaluate_delphi_fitting(region, start_date_default, end_date_default, split_default, results_df)
    except ValueError as err:
        print(f"Error in evaluating region {region}, error: {err}")

Evaluating Delphi for US, New York
Finished predicting for Country=US and Province=New York in 109.0 seconds
Evaluating Delphi for US, Florida


/var/folders/9v/_4c_8bdj3x7g3plwm_sjtm3m0000gn/T/ipykernel_34108/1326276864.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  totalcases.date = pd.to_datetime(totalcases.date).dt.strftime('%Y-%m-%d')
/Users/saksham/Library/Mobile Documents/com~apple~CloudDocs/Workspace/Research/THEMIS/code/THEMIS/pandemic_functions/delphi_functions/DELPHI_model_fitting.py:216: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  totalcases.date = pd.to_datetime(totalcases.date).dt.strftime('%Y-%m-%d')


Finished predicting for Country=US and Province=Florida in 112.28 seconds
Evaluating Delphi for Spain, None
Finished predicting for Country=Spain and Province=None in 307.03 seconds
Evaluating Delphi for Brazil, None
Finished predicting for Country=Brazil and Province=None in 100.62 seconds
Evaluating Delphi for Singapore, None
Finished predicting for Country=Singapore and Province=None in 482.0 seconds
Evaluating Delphi for Germany, None
Finished predicting for Country=Germany and Province=None in 99.85 seconds


In [19]:
results_df

,region,start_date,split_date,end_date,target,MSE,MAE,MAPE
0,US-NY,2020-03-08,2020-04-08,2020-06-08,Total Detected Cases,15884494848.225807,111475.709677,32.667001
1,US-NY,2020-03-08,2020-04-08,2020-06-08,Total Detected Deaths,1196136892.370968,28391.241935,99.155468
2,US-FL,2020-03-15,2020-04-15,2020-06-15,Total Detected Cases,336918419.096774,13932.677419,25.253902
3,US-FL,2020-03-15,2020-04-15,2020-06-15,Total Detected Deaths,242818.693548,376.758065,16.277937
4,ES,2020-03-02,2020-04-02,2020-06-02,Total Detected Cases,16000089387.016129,113599.403226,51.793788
5,ES,2020-03-02,2020-04-02,2020-06-02,Total Detected Deaths,653704960.016129,22407.241935,88.483779
6,BR,2020-03-13,2020-04-13,2020-06-13,Total Detected Cases,106613337224.5,226309.435484,56.933013
7,BR,2020-03-13,2020-04-13,2020-06-13,Total Detected Deaths,214907919.758065,10364.983871,43.077039
8,SG,2020-03-01,2020-04-01,2020-06-01,Total Detected Cases,326773901.0,14392.83871,67.663512
9,SG,2020-03-01,2020-04-01,2020-06-01,Total Detected Deaths,5.806452,1.870968,11.201798


### Save results

In [22]:
results_df = results_df.pivot(index=['region', 'start_date', 'split_date', 'end_date'], 
                 columns='target', values=['MSE', 'MAE', 'MAPE'])

In [23]:
results_df

MSE   
target                                  Total Detected Cases   
region start_date split_date end_date                          
BR     2020-03-13 2020-04-13 2020-06-13       106613337224.5  \
DE     2020-03-01 2020-04-01 2020-06-01   10220750781.951612   
ES     2020-03-02 2020-04-02 2020-06-02   16000089387.016129   
SG     2020-03-01 2020-04-01 2020-06-01          326773901.0   
US-FL  2020-03-15 2020-04-15 2020-06-15     336918419.096774   
US-NY  2020-03-08 2020-04-08 2020-06-08   15884494848.225807   

                                                                
target                                  Total Detected Deaths   
region start_date split_date end_date                           
BR     2020-03-13 2020-04-13 2020-06-13      214907919.758065  \
DE     2020-03-01 2020-04-01 2020-06-01         674887.016129   
ES     2020-03-02 2020-04-02 2020-06-02      653704960.016129   
SG     2020-03-01 2020-04-01 2020-06-01              5.806452   
US-FL  2020-03-15 2020-04-15 2020-06-15         242818.693548   
US-NY  2020-03-08 2020-04-08 2020-06-08     1196136892.370968   

                                                         MAE   
target                                  Total Detected Cases   
region start_date split_date end_date                          
BR     2020-03-13 2020-04-13 2020-06-13        226309.435484  \
DE     2020-03-01 2020-04-01 2020-06-01         89716.822581   
ES     2020-03-02 2020-04-02 2020-06-02        113599.403226   
SG     2020-03-01 2020-04-01 2020-06-01          14392.83871   
US-FL  2020-03-15 2020-04-15 2020-06-15         13932.677419   
US-NY  2020-03-08 2020-04-08 2020-06-08        111475.709677   

                                                                
target                                  Total Detected Deaths   
region start_date split_date end_date                           
BR     2020-03-13 2020-04-13 2020-06-13          10364.983871  \
DE     2020-03-01 2020-04-01 2020-06-01            718.693548   
ES     2020-03-02 2020-04-02 2020-06-02          22407.241935   
SG     2020-03-01 2020-04-01 2020-06-01              1.870968   
US-FL  2020-03-15 2020-04-15 2020-06-15            376.758065   
US-NY  2020-03-08 2020-04-08 2020-06-08          28391.241935   

                                                        MAPE   
target                                  Total Detected Cases   
region start_date split_date end_date                          
BR     2020-03-13 2020-04-13 2020-06-13            56.933013  \
DE     2020-03-01 2020-04-01 2020-06-01            53.837519   
ES     2020-03-02 2020-04-02 2020-06-02            51.793788   
SG     2020-03-01 2020-04-01 2020-06-01            67.663512   
US-FL  2020-03-15 2020-04-15 2020-06-15            25.253902   
US-NY  2020-03-08 2020-04-08 2020-06-08            32.667001   

                                                               
target                                  Total Detected Deaths  
region start_date split_date end_date                          
BR     2020-03-13 2020-04-13 2020-06-13             43.077039  
DE     2020-03-01 2020-04-01 2020-06-01             13.694773  
ES     2020-03-02 2020-04-02 2020-06-02             88.483779  
SG     2020-03-01 2020-04-01 2020-06-01             11.201798  
US-FL  2020-03-15 2020-04-15 2020-06-15             16.277937  
US-NY  2020-03-08 2020-04-08 2020-06-08             99.155468

In [24]:
results_df.to_csv('notebooks/DELPHI_performance_w_1_month_training.csv')